<a href="https://colab.research.google.com/github/SolKacil/matematicas-para-ia/blob/main/python/01-algebra-lineal/01_vectores_y_espacios_vectoriales.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# 01 &middot; Vectores y espacios vectoriales

**Módulo 1 — Álgebra lineal y geometría diferencial**

Este notebook acompaña al documento `Matematicas_para_IA_01.pdf`, basado en Deisenroth, Faisal y
Ong (2020), *Mathematics for Machine Learning*, sección 2.4.

El notebook anterior definió operaciones sobre matrices; éste identifica la **estructura** que esas
operaciones forman. La pregunta que organiza el tema es de qué depende que una expresión como
$\boldsymbol{\theta}_{t+1} = \boldsymbol{\theta}_t - \eta\nabla L(\boldsymbol{\theta}_t)$ esté bien
definida. La respuesta es que el conjunto donde viven los parámetros sea un **espacio vectorial**:
sin esa estructura, restar dos conjuntos de parámetros o multiplicarlos por una tasa de aprendizaje
no significaría nada.

El desarrollo va de lo general a lo particular: primero **grupos**, que exigen únicamente una
operación con neutro e inversos; después **espacios vectoriales**, que añaden la multiplicación por
escalares; finalmente **subespacios**, que son los subconjuntos que preservan la estructura y
constituyen el objeto que la reducción de dimensionalidad manipula.

Una advertencia sobre el método. Verificar un axioma con ejemplos numéricos **no lo demuestra**: una
propiedad universal no se establece por muestreo. El código de este notebook sirve para dos cosas
distintas y legítimas: comprobar que un caso concreto es consistente con lo que afirma la teoría, y
—lo más útil— **construir contraejemplos**, que sí son concluyentes, porque para refutar una
afirmación universal basta un caso.

## Al terminar será posible

- Verificar los cuatro axiomas de **grupo** sobre un conjunto y una operación dados, y detectar cuál
  falla cuando no se cumplen.
- Reconocer el **grupo lineal general** $GL(n, \mathbb{R})$ y justificar por qué las matrices
  singulares quedan excluidas.
- Enunciar los axiomas de **espacio vectorial** y comprobarlos numéricamente sobre $\mathbb{R}^n$ y
  $\mathbb{R}^{m \times n}$.
- Aplicar el **criterio práctico de subespacio** —contener el cero y ser cerrado bajo suma y
  producto por escalar— y construir contraejemplos cuando falla.
- Identificar $N(A)$ como subespacio y el conjunto solución de $A\mathbf{x} = \mathbf{b}$ con
  $\mathbf{b} \neq \mathbf{0}$ como un conjunto que no lo es.

## Qué se da por sabido

El notebook [00 · Propiedades de matrices](00_propiedades_matrices.ipynb). Los conceptos de
determinante y de matriz invertible se usan de manera auxiliar en la sección 2; se desarrollan en el
notebook 03.

## Cómo usar este notebook

1. Con el botón **Open in Colab** no se requiere instalación alguna.
2. Las celdas se ejecutan en orden con `Shift + Enter`.
3. Ante cada afirmación conviene preguntarse **qué habría que encontrar para refutarla**, y después
   buscarlo con código: es la forma más rápida de entender un axioma.
4. La sección final, **Tu turno**, contiene los ejercicios propuestos del documento.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from itertools import product

np.set_printoptions(precision=4, suppress=True)
plt.rcParams["figure.figsize"] = (5.5, 5.5)

print("numpy:", np.__version__)

---

## 1. Grupos

### 1.1 Definición

**Notación.** Las letras caligráficas como $\mathcal{G}$ denotan conjuntos abstractos, cuyos
elementos pueden ser números, vectores, matrices o cualquier otro objeto. El símbolo $\forall$ se
lee «para todo» y $\exists$ «existe»; $x \in \mathcal{G}$ se lee «$x$ pertenece a $\mathcal{G}$».
El símbolo $\otimes$ representa una operación genérica, no necesariamente una multiplicación: puede
ser una suma o cualquier regla que combine dos elementos del conjunto para producir otro elemento
del mismo conjunto.

**Definición (grupo).** Sea $\mathcal{G}$ un conjunto y $\otimes : \mathcal{G} \times \mathcal{G}
\to \mathcal{G}$ una operación. El par $G := (\mathcal{G}, \otimes)$ es un **grupo** si:

1. **Cerradura:** $\forall x, y \in \mathcal{G} : x \otimes y \in \mathcal{G}$
2. **Asociatividad:** $\forall x, y, z \in \mathcal{G} : (x \otimes y) \otimes z = x \otimes (y \otimes z)$
3. **Elemento neutro:** $\exists e \in \mathcal{G} \; \forall x \in \mathcal{G} : x \otimes e = x = e \otimes x$
4. **Elemento inverso:** $\forall x \in \mathcal{G} \; \exists y \in \mathcal{G} : x \otimes y = e = y \otimes x$,
   denotado $x^{-1}$

Si además $\forall x, y \in \mathcal{G} : x \otimes y = y \otimes x$, el grupo es **abeliano** o
conmutativo.

El inverso se define **respecto a la operación** $\otimes$; no significa necesariamente $1/x$. En
$(\mathbb{Z}, +)$ el inverso de $3$ es $-3$, no $1/3$.

La función siguiente verifica los cuatro axiomas por enumeración exhaustiva. Sólo es aplicable a
conjuntos finitos, pero en ese caso su veredicto es concluyente.

In [ ]:
def verificar_grupo(elementos, op, nombre="G"):
    """Comprueba los cuatro axiomas de grupo por enumeracion sobre un conjunto finito."""
    conjunto = list(elementos)
    pertenece = set(conjunto)

    cerradura = all(op(x, y) in pertenece for x in conjunto for y in conjunto)
    asociativa = all(op(op(x, y), z) == op(x, op(y, z))
                     for x in conjunto for y in conjunto for z in conjunto)

    neutros = [e for e in conjunto if all(op(x, e) == x == op(e, x) for x in conjunto)]
    e = neutros[0] if neutros else None

    if e is None:
        inversos, sin_inverso = False, conjunto
    else:
        sin_inverso = [x for x in conjunto
                       if not any(op(x, y) == e == op(y, x) for y in conjunto)]
        inversos = not sin_inverso

    abeliano = all(op(x, y) == op(y, x) for x in conjunto for y in conjunto)
    es_grupo = cerradura and asociativa and e is not None and inversos

    print(f"{nombre}: cerradura={cerradura}  asociativa={asociativa}  "
          f"neutro={e}  inversos={inversos}")
    if sin_inverso and e is not None:
        print(f"   sin inverso: {sin_inverso[:5]}")
    print(f"   -> {'grupo abeliano' if es_grupo and abeliano else 'grupo' if es_grupo else 'NO es grupo'}")
    return es_grupo


# Los enteros modulo 5 con la suma: un grupo finito, verificable por completo.
verificar_grupo(range(5), lambda x, y: (x + y) % 5, "(Z_5, +)")

# Los mismos elementos con la multiplicacion modulo 5: el 0 no tiene inverso.
verificar_grupo(range(5), lambda x, y: (x * y) % 5, "(Z_5, *)")

# Al excluir el 0, si es grupo (5 es primo).
verificar_grupo(range(1, 5), lambda x, y: (x * y) % 5, "(Z_5 \\ {0}, *)")

### 1.2 Ejemplos

$\mathbb{Z}$ denota los números enteros, $\mathbb{N}_0$ los naturales incluyendo el cero y
$\mathbb{R}$ los reales. Para conjuntos infinitos la enumeración no es posible, pero los
contraejemplos sí se pueden exhibir, y son lo único que hace falta para refutar.

| Conjunto y operación | ¿Grupo? | Motivo |
|---|---|---|
| $(\mathbb{Z}, +)$ | abeliano | neutro $0$; inverso de $x$ es $-x$ |
| $(\mathbb{N}_0, +)$ | **no** | tiene neutro, faltan inversos: $-3 \notin \mathbb{N}_0$ |
| $(\mathbb{Z}, \cdot)$ | **no** | tiene neutro $1$, faltan inversos: $1/2 \notin \mathbb{Z}$ |
| $(\mathbb{R} \setminus \{0\}, \cdot)$ | abeliano | se excluye el único elemento sin inverso |
| $(\mathbb{R}^n, +)$ | abeliano | suma componente a componente |
| $(\mathbb{R}^{m \times n}, +)$ | abeliano | suma entrada a entrada |
| $(\mathbb{R}^{n \times n}, \cdot)$ | **no** | las matrices singulares no tienen inverso |

In [ ]:
# (N_0, +): el neutro existe, pero 3 no tiene inverso dentro del conjunto.
candidatos = [y for y in range(0, 1000) if 3 + y == 0]
print("(N_0, +): inversos de 3 encontrados en {0, ..., 999}:", candidatos, "-> ninguno")
print("          el inverso seria -3, que no pertenece a N_0")

# (Z, .): el inverso multiplicativo de 2 seria 0.5, que no es entero.
print("\n(Z, .):  2 * y = 1 exige y = 0.5 ->", 2 * 0.5 == 1, ", pero 0.5 no es entero")

# (R^n, +): abeliano. Se comprueba sobre un caso concreto.
x = np.array([1.0, 3.0])
y = np.array([2.0, -1.0])
print("\n(R^2, +):  x + y =", x + y, " y + x =", y + x, " iguales:", np.allclose(x + y, y + x))
print("          neutro:", np.zeros(2), "  inverso de x:", -x, " x + (-x) =", x + (-x))

In [ ]:
# (R^{n x n}, .): cerradura y asociatividad se cumplen; el neutro es la identidad.
A = np.array([[1.0, 2.0],
              [0.0, 1.0]])
B = np.array([[2.0, 0.0],
              [1.0, 3.0]])
C = np.array([[1.0, 1.0],
              [2.0, 0.0]])

print("(A@B)@C =\n", (A @ B) @ C)
print("\nA@(B@C) =\n", A @ (B @ C))
print("\nasociativa:", np.allclose((A @ B) @ C, A @ (B @ C)))
print("neutro I2 :", np.allclose(A @ np.eye(2), A) and np.allclose(np.eye(2) @ A, A))

# Pero el axioma de inverso falla: hay matrices sin inversa.
singular = np.array([[1.0, 2.0],
                     [2.0, 4.0]])
print("\ndet de la matriz singular:", np.linalg.det(singular))
try:
    np.linalg.inv(singular)
except np.linalg.LinAlgError as e:
    print("inv(singular) ->", e)
print("-> (R^{2x2}, .) NO es grupo: falta el axioma 4")

### 1.3 El grupo lineal general

**Definición.** El conjunto de las matrices invertibles $A \in \mathbb{R}^{n \times n}$ es un grupo
respecto a la multiplicación de matrices, llamado **grupo lineal general** $GL(n, \mathbb{R})$.

Excluir las matrices singulares es exactamente la corrección que restituye el axioma que faltaba. La
cerradura se conserva: si $\det(A) \neq 0$ y $\det(B) \neq 0$, entonces
$\det(AB) = \det(A)\det(B) \neq 0$, de modo que el producto de dos matrices invertibles es
invertible.

El grupo **no es abeliano**, porque la multiplicación de matrices no lo es.

In [ ]:
print("A @ B =\n", A @ B)
print("\nB @ A =\n", B @ A)
print("\nconmutan:", np.allclose(A @ B, B @ A), " -> GL(2, R) no es abeliano")

# Cerradura: det(AB) = det(A) det(B), luego el producto de invertibles es invertible.
print(f"\ndet(A) = {np.linalg.det(A):.4f}   det(B) = {np.linalg.det(B):.4f}")
print(f"det(A@B) = {np.linalg.det(A @ B):.4f}   det(A)*det(B) = "
      f"{np.linalg.det(A) * np.linalg.det(B):.4f}")

# Inverso y neutro dentro del grupo.
print("\nA @ inv(A) =\n", A @ np.linalg.inv(A))

rng = np.random.default_rng(0)
dets = [np.linalg.det(rng.normal(size=(3, 3)) @ rng.normal(size=(3, 3))) for _ in range(5)]
print("\ndeterminantes de productos de matrices genericas:", np.round(dets, 4))
print("ninguno es cero: el producto se mantiene dentro de GL(3, R)")

### Lectura en aprendizaje automático

Los *normalizing flows* son una familia de modelos generativos que construyen una distribución
compleja aplicando una secuencia de transformaciones **invertibles** a una distribución simple. Para
evaluar la densidad de probabilidad resultante el modelo necesita poder deshacer exactamente cada
transformación, lo que obliga a que cada matriz de pesos pertenezca a $GL(n, \mathbb{R})$ y no
simplemente a $\mathbb{R}^{n \times n}$.

La distinción es operativa, no terminológica: al diseñar la arquitectura hay que **garantizar** la
invertibilidad por construcción —por ejemplo parametrizando la matriz mediante una factorización
$LU$ con diagonal no nula, o restringiéndola a ser ortogonal, como en la sección 4 del notebook 03—
en lugar de confiar en que una matriz arbitraria resulte invertible.

In [ ]:
def capa_invertible(x, W, b):
    return x @ W + b


def capa_inversa(y, W, b):
    return (y - b) @ np.linalg.inv(W)


rng = np.random.default_rng(2)
W = rng.normal(size=(3, 3))
b = rng.normal(size=3)
x = rng.normal(size=(4, 3))

y = capa_invertible(x, W, b)
x_recuperado = capa_inversa(y, W, b)

print(f"det(W) = {np.linalg.det(W):.4f}  -> W pertenece a GL(3, R)")
print("se recupera la entrada:", np.allclose(x, x_recuperado))
print(f"error maximo: {np.abs(x - x_recuperado).max():.2e}")

# Con una matriz singular la transformacion no se puede deshacer.
W_sing = W.copy()
W_sing[2, :] = W_sing[0, :]          # dos filas iguales -> det = 0
print(f"\ndet(W singular) = {np.linalg.det(W_sing):.2e}   rango = {np.linalg.matrix_rank(W_sing)} < 3")

try:
    recuperado = capa_inversa(capa_invertible(x, W_sing, b), W_sing, b)
    print(f"inv() no lanzo error, pero el error de reconstruccion es "
          f"{np.abs(x - recuperado).max():.2e}")
except np.linalg.LinAlgError as e:
    print("no se puede invertir la capa ->", e)

# El determinante no es exactamente cero por el redondeo de la aritmetica de punto
# flotante, asi que inv() puede no protestar y devolver un resultado sin sentido.
# El criterio fiable es el numero de condicion, no el determinante.
print(f"numero de condicion: {np.linalg.cond(W_sing):.2e}  <- la senal de que W no es invertible")

# Ademas, la capa singular pierde informacion: dos entradas distintas dan la misma salida.
h = np.array([1.0, 0.0, -1.0])       # como la fila 2 es igual a la fila 0, h @ W_sing = 0
print("\nh @ W_sing =", h @ W_sing)
print("dos entradas distintas, la misma salida:",
      np.allclose(capa_invertible(x, W_sing, b), capa_invertible(x + h, W_sing, b)))

---

## 2. Espacios vectoriales

### 2.1 Definición

**Definición (espacio vectorial real).** Un espacio vectorial real $V = (\mathcal{V}, +, \cdot)$ es
un conjunto $\mathcal{V}$ con dos operaciones, $+ : \mathcal{V} \times \mathcal{V} \to \mathcal{V}$
y $\cdot : \mathbb{R} \times \mathcal{V} \to \mathcal{V}$, tales que:

1. $(\mathcal{V}, +)$ es un **grupo abeliano**.
2. **Distributividad:**
   $\lambda \cdot (\mathbf{x} + \mathbf{y}) = \lambda \cdot \mathbf{x} + \lambda \cdot \mathbf{y}$ y
   $(\lambda + \psi) \cdot \mathbf{x} = \lambda \cdot \mathbf{x} + \psi \cdot \mathbf{x}$.
3. **Asociatividad de la operación externa:**
   $\lambda \cdot (\psi \cdot \mathbf{x}) = (\lambda\psi) \cdot \mathbf{x}$.
4. **Neutro externo:** $1 \cdot \mathbf{x} = \mathbf{x}$.

Los elementos $\mathbf{x} \in V$ son **vectores**; el neutro de $(\mathcal{V}, +)$ es el **vector
cero**, denotado $\mathbf{0}$ en negritas para distinguirlo del escalar $0$; los $\lambda \in
\mathbb{R}$ son **escalares**.

Obsérvese que la definición no menciona ninguna «multiplicación de vectores». No forma parte de la
estructura, y de hecho el producto elemento a elemento —habitual en programación— no corresponde a
ninguna operación del álgebra lineal. Viendo los vectores como matrices $n \times 1$, lo que sí está
definido es:

$$\mathbf{a}\mathbf{b}^\top \in \mathbb{R}^{n \times n} \quad \text{(producto externo)}, \qquad
\mathbf{a}^\top\mathbf{b} \in \mathbb{R} \quad \text{(producto interno o punto)}.$$

In [ ]:
x = np.array([1.0, 2.0])
y = np.array([3.0, -1.0])
lam, psi = 2.0, 3.0

print("distributividad  lam(x+y) == lam*x + lam*y :",
      np.allclose(lam * (x + y), lam * x + lam * y), "  ->", lam * (x + y))
print("distributividad  (lam+psi)x == lam*x + psi*x:",
      np.allclose((lam + psi) * x, lam * x + psi * x), "  ->", (lam + psi) * x)
print("asociatividad    lam(psi*x) == (lam*psi)x   :",
      np.allclose(lam * (psi * x), (lam * psi) * x), "  ->", lam * (psi * x))
print("neutro externo   1*x == x                   :", np.allclose(1.0 * x, x))

# (V, +) es grupo abeliano: neutro, inverso y conmutatividad.
print("\nneutro:", np.zeros(2), "  inverso de x:", -x, "  x + (-x) =", x + (-x))
print("conmutativa:", np.allclose(x + y, y + x))

In [ ]:
a = np.array([1.0, 2.0, 3.0])
b = np.array([4.0, 0.0, -1.0])

print("producto interno  a.T @ b =", a @ b, "  forma:", np.shape(a @ b), "(un escalar)")
print("\nproducto externo  a @ b.T =\n", np.outer(a, b), "\nforma:", np.outer(a, b).shape)

print("\nelemento a elemento a * b =", a * b)
print("<- operacion valida en NumPy, pero NO es una operacion de espacio vectorial")

### 2.2 Ejemplos

- $V = \mathbb{R}^n$, con suma y producto por escalar componente a componente.
- $V = \mathbb{R}^{m \times n}$, con ambas operaciones entrada a entrada. Como espacio vectorial es
  indistinguible de $\mathbb{R}^{mn}$: sólo cambia la disposición de las componentes.
- $V = \mathbb{C}$, con la suma estándar de números complejos.

La equivalencia entre $\mathbb{R}^{m \times n}$ y $\mathbb{R}^{mn}$ no es una curiosidad formal: es
la razón por la que un optimizador puede tratar todos los parámetros de una red —matrices de pesos
de formas distintas, vectores de sesgo— como un único vector largo, y por la que `reshape` no altera
nada esencial.

In [ ]:
W1 = np.array([[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]])     # 3x2
W2 = np.array([[10.0, 20.0], [30.0, 40.0], [50.0, 60.0]])

# Las operaciones de espacio vectorial sobre matrices actuan entrada a entrada.
print("W1 + W2 =\n", W1 + W2)
print("\n0.5 * W1 =\n", 0.5 * W1)

# Y son las mismas que sobre el vector aplanado de 6 componentes.
v1, v2 = W1.ravel(), W2.ravel()
print("\naplanadas:", v1, "y", v2)
print("suma aplanada == aplanado de la suma:", np.allclose(v1 + v2, (W1 + W2).ravel()))

# Asi es como un optimizador trata todos los parametros de un modelo a la vez.
parametros = [np.zeros((3, 2)), np.zeros(4), np.zeros((2, 2))]
plano = np.concatenate([p.ravel() for p in parametros])
print(f"\n3 tensores de formas {[p.shape for p in parametros]} -> un vector de {plano.size} componentes")

### Lectura en aprendizaje automático

La regla de actualización del descenso de gradiente,

$$\boldsymbol{\theta}_{t+1} = \boldsymbol{\theta}_t - \eta\,\nabla L(\boldsymbol{\theta}_t),$$

es literalmente una combinación de las dos operaciones de espacio vectorial: una multiplicación por
el escalar $\eta$ y una resta de vectores. Sin la estructura de espacio vectorial sobre el conjunto
de parámetros, la expresión no estaría definida.

La misma observación justifica el **promediado de modelos**. Como $\mathbb{R}^{m \times n}$ es un
espacio vectorial, la combinación $W_{\text{prom}} = \frac{1}{k}(W_1 + \dots + W_k)$ es un elemento
del mismo espacio, y por tanto un modelo válido. En eso se apoyan el aprendizaje federado y las
*model soups*. Conviene subrayar el alcance exacto del argumento: la estructura garantiza que el
promedio **sea un modelo bien definido**, no que sea un modelo bueno. Que además funcione depende de
que los modelos promediados estén conectados en la región del espacio de parámetros donde la pérdida
es baja, lo cual es una cuestión empírica ajena al álgebra.

In [ ]:
def descenso(theta, gradiente, eta):
    """Un paso de descenso: producto por escalar y resta de vectores."""
    return theta - eta * gradiente


theta = np.array([1.0, -2.0, 0.5])
grad = np.array([0.4, 0.1, -0.3])

print("theta          =", theta)
print("gradiente      =", grad)
print("theta - 0.1*g  =", descenso(theta, grad, 0.1))

# La misma operacion sobre una lista de tensores de formas distintas.
rng = np.random.default_rng(1)
pesos = [rng.normal(size=(2, 3)), rng.normal(size=3)]
grads = [rng.normal(size=(2, 3)), rng.normal(size=3)]
nuevos = [descenso(p, g, 0.1) for p, g in zip(pesos, grads)]
print("\nformas conservadas tras el paso:", [p.shape for p in nuevos])

# Promediar modelos: otra operacion de espacio vectorial.
modelos = [rng.normal(size=(2, 2)) for _ in range(4)]
print("\npromedio de 4 modelos =\n", sum(modelos) / len(modelos))

---

## 3. Subespacios vectoriales

### 3.1 Definición y criterio práctico

**Notación.** $U \subseteq V$ se lee «$U$ es subconjunto de $V$»; $\emptyset$ denota el conjunto
vacío.

**Definición (subespacio vectorial).** Sea $V = (\mathcal{V}, +, \cdot)$ un espacio vectorial y
$\mathcal{U} \subseteq \mathcal{V}$ con $\mathcal{U} \neq \emptyset$. Entonces
$U = (\mathcal{U}, +, \cdot)$ es un **subespacio** de $V$ si $U$ es un espacio vectorial con las
operaciones de $V$ restringidas a $\mathcal{U}$.

Verificar todos los axiomas sería redundante, porque $U$ hereda de $V$ la asociatividad, la
conmutatividad, la distributividad y el neutro externo. En la práctica basta comprobar tres
condiciones:

1. $\mathbf{0} \in \mathcal{U}$ (en particular $\mathcal{U} \neq \emptyset$).
2. **Cerradura bajo el producto por escalar:** $\forall \lambda \in \mathbb{R}, \forall \mathbf{x} \in \mathcal{U} : \lambda\mathbf{x} \in \mathcal{U}$.
3. **Cerradura bajo la suma:** $\forall \mathbf{x}, \mathbf{y} \in \mathcal{U} : \mathbf{x} + \mathbf{y} \in \mathcal{U}$.

La función siguiente aplica el criterio por muestreo. Su utilidad es asimétrica y conviene tenerlo
presente: cuando encuentra un contraejemplo, **demuestra** que el conjunto no es subespacio; cuando
no lo encuentra, únicamente indica que no lo halló entre las muestras probadas.

In [ ]:
def criterio_subespacio(pertenece, muestras, escalares=(-2.0, -0.5, 0.0, 3.0), nombre="U"):
    """Aplica por muestreo las tres condiciones del criterio practico.

    `pertenece(v)` decide si un vector esta en el conjunto; `muestras` es una
    lista de vectores del conjunto. Un contraejemplo es concluyente; su ausencia no.
    """
    dim = len(muestras[0])
    tiene_cero = bool(pertenece(np.zeros(dim)))

    fallo_escalar = next((( v, lam) for v in muestras for lam in escalares
                          if not pertenece(lam * np.asarray(v, dtype=float))), None)
    fallo_suma = next(((u, v) for u in muestras for v in muestras
                       if not pertenece(np.asarray(u, dtype=float) + np.asarray(v, dtype=float))),
                      None)

    print(f"{nombre}:")
    print(f"   contiene 0            : {tiene_cero}")
    if fallo_escalar is None:
        print("   cerrado bajo escalar  : no se hallo contraejemplo")
    else:
        v, lam = fallo_escalar
        print(f"   cerrado bajo escalar  : NO  ({lam} * {np.asarray(v)} sale del conjunto)")
    if fallo_suma is None:
        print("   cerrado bajo suma     : no se hallo contraejemplo")
    else:
        u, v = fallo_suma
        print(f"   cerrado bajo suma     : NO  ({np.asarray(u)} + {np.asarray(v)} sale del conjunto)")

    veredicto = tiene_cero and fallo_escalar is None and fallo_suma is None
    print(f"   -> {'compatible con ser subespacio' if veredicto else 'NO es subespacio'}")
    return veredicto

### 3.2 Ejemplos

Se consideran cuatro subconjuntos de $\mathbb{R}^2$:

- $A = \{(x,y) : 0 \leq x \leq 1,\; 0 \leq y \leq 1\}$, el cuadrado unitario.
- $B = \{(x,y) : y = x + 1\}$, una recta que no pasa por el origen.
- $C = \{(x,y) : x \geq 0,\; y \geq 0\}$, el primer cuadrante.
- $D = \{(x,y) : y = 2x\}$, una recta que pasa por el origen.

Sólo $D$ es subespacio. Los otros tres fallan por motivos distintos, y conviene identificar cuál en
cada caso: $A$ no es cerrado bajo escalares mayores que uno, $B$ no contiene al origen y $C$ no es
cerrado bajo escalares negativos.

In [ ]:
conjuntos = {
    "A (cuadrado unitario)": (lambda v: 0 <= v[0] <= 1 and 0 <= v[1] <= 1,
                              [(1.0, 1.0), (0.5, 0.25), (0.0, 1.0)]),
    "B (recta y = x + 1)": (lambda v: np.isclose(v[1], v[0] + 1),
                            [(0.0, 1.0), (2.0, 3.0), (-1.0, 0.0)]),
    "C (primer cuadrante)": (lambda v: v[0] >= 0 and v[1] >= 0,
                             [(1.0, 1.0), (2.0, 0.5), (0.0, 3.0)]),
    "D (recta y = 2x)": (lambda v: np.isclose(v[1], 2 * v[0]),
                         [(1.0, 2.0), (-3.0, -6.0), (0.5, 1.0)]),
}

for nombre, (pertenece, muestras) in conjuntos.items():
    criterio_subespacio(pertenece, muestras, nombre=nombre)
    print()

In [ ]:
fig, ejes = plt.subplots(1, 4, figsize=(15, 4))
t = np.linspace(-3, 3, 100)

# A: cuadrado unitario. Contraejemplo: 2*(1,1) = (2,2) se sale.
ax = ejes[0]
ax.fill([0, 1, 1, 0], [0, 0, 1, 1], color="tab:blue", alpha=0.3)
ax.plot([1], [1], "o", color="tab:blue")
ax.annotate("(1,1)", (1, 1), textcoords="offset points", xytext=(-38, -4), fontsize=8)
ax.plot([2], [2], "X", color="tab:red", ms=9)
ax.annotate("2(1,1)", (2, 2), textcoords="offset points", xytext=(4, -4),
            color="tab:red", fontsize=8)
ax.set_title("A: cuadrado\nno cerrado bajo escalar", fontsize=9)

# B: recta que no pasa por el origen.
ax = ejes[1]
ax.plot(t, t + 1, color="tab:blue")
ax.plot([0], [0], "X", color="tab:red", ms=9)
ax.annotate("0 no esta", (0, 0), textcoords="offset points", xytext=(6, -12),
            color="tab:red", fontsize=8)
ax.set_title("B: y = x + 1\nno contiene el origen", fontsize=9)

# C: primer cuadrante. Contraejemplo: -1*(1,1) = (-1,-1).
ax = ejes[2]
ax.fill([0, 3, 3, 0], [0, 0, 3, 3], color="tab:blue", alpha=0.3)
ax.plot([1], [1], "o", color="tab:blue")
ax.plot([-1], [-1], "X", color="tab:red", ms=9)
ax.annotate("-1(1,1)", (-1, -1), textcoords="offset points", xytext=(-14, -14),
            color="tab:red", fontsize=8)
ax.set_title("C: primer cuadrante\nno cerrado bajo escalar < 0", fontsize=9)

# D: recta por el origen. Si es subespacio.
ax = ejes[3]
ax.plot(t, 2 * t, color="tab:green")
ax.plot([0], [0], "o", color="tab:green")
ax.plot([1, -1.5], [2, -3], "o", color="tab:green", ms=5)
ax.set_title("D: y = 2x\nsi es subespacio", fontsize=9)

for ax in ejes:
    ax.set_xlim(-3, 3); ax.set_ylim(-3, 3)
    ax.axhline(0, color="gray", lw=0.8); ax.axvline(0, color="gray", lw=0.8)
    ax.grid(alpha=0.3); ax.set_aspect("equal")
plt.show()

### 3.3 Tres casos que conviene tener presentes

**El espacio nulo $N(A)$ siempre es subespacio.** Contiene a $\mathbf{0}$, porque
$A\mathbf{0} = \mathbf{0}$, y es cerrado bajo ambas operaciones: si
$\mathbf{x}_h, \mathbf{x}_h' \in N(A)$, entonces

$$A(\lambda_1\mathbf{x}_h + \lambda_2\mathbf{x}_h') = \lambda_1 A\mathbf{x}_h + \lambda_2 A\mathbf{x}_h'
= \lambda_1\mathbf{0} + \lambda_2\mathbf{0} = \mathbf{0}.$$

**El conjunto solución de $A\mathbf{x} = \mathbf{b}$ con $\mathbf{b} \neq \mathbf{0}$ no lo es.**
Falla por partida doble: no contiene al origen y no es cerrado bajo la suma. Es, como se verá en el
notebook 02, una **traslación** de $N(A)$ —un subespacio afín, no un subespacio vectorial.

**La intersección de subespacios es subespacio.** Si $U_1 = \{(x,y,z) : x = 0\}$ y
$U_2 = \{(x,y,z) : y = 0\}$ en $\mathbb{R}^3$, entonces $U_1 \cap U_2$ es el eje $z$, una recta por
el origen.

In [ ]:
A = np.array([[1.0, 2.0],
              [2.0, 4.0]])

# N(A) = { t(-2, 1) }: cerrado bajo combinaciones lineales.
h1 = np.array([-2.0, 1.0])
h2 = 3.0 * h1
print("A @ h1 =", A @ h1, "   A @ h2 =", A @ h2)
print("A @ (2*h1 - 5*h2) =", A @ (2 * h1 - 5 * h2), " <- sigue en N(A)")

# El conjunto solucion de A x = b con b distinto de cero no es cerrado.
b = np.array([1.0, 2.0])
x1 = np.array([1.0, 0.0])
x2 = x1 + h1                       # = (-1, 1), tambien solucion
print(f"\nA @ x1 = {A @ x1}   A @ x2 = {A @ x2}   b = {b}")
print("x1 + x2 =", x1 + x2, "  A @ (x1 + x2) =", A @ (x1 + x2), " != b")
print("ademas, A @ 0 =", A @ np.zeros(2), "!= b, luego el origen tampoco pertenece")

In [ ]:
# La interseccion de dos subespacios de R^3 es subespacio (aqui, el eje z).
en_U1 = lambda v: np.isclose(v[0], 0.0)                      # x = 0
en_U2 = lambda v: np.isclose(v[1], 0.0)                      # y = 0
en_interseccion = lambda v: en_U1(v) and en_U2(v)

criterio_subespacio(en_interseccion,
                    [(0.0, 0.0, 1.0), (0.0, 0.0, -4.0), (0.0, 0.0, 2.5)],
                    nombre="U1 ∩ U2 (eje z)")

# La union, en cambio, no lo es: (1,0,0) y (0,1,0) estan, pero su suma no.
en_union = lambda v: en_U1(v) or en_U2(v)
print()
criterio_subespacio(en_union, [(0.0, 1.0, 0.0), (1.0, 0.0, 0.0)], nombre="U1 U U2 (union)")

### Lectura en aprendizaje automático

La reducción de dimensionalidad es, en esencia, la sustitución de $\mathbb{R}^n$ por un subespacio
de dimensión $k < n$. El **análisis de componentes principales** busca el subespacio de dimensión
$k$ que mejor captura la variabilidad de los datos y proyecta cada observación sobre él.

Que el objetivo sea un subespacio —y no un subconjunto cualquiera— tiene una consecuencia práctica
inmediata: la proyección es una transformación **lineal**, representable por una matriz, y las
combinaciones lineales de datos proyectados siguen perteneciendo al subespacio. Sobre eso descansa
la posibilidad de operar directamente en el espacio reducido: promediar representaciones, calcular
distancias, interpolar entre puntos.

Obsérvese también que el subespacio pasa por el origen, lo cual explica por qué PCA exige **centrar
los datos** antes de proyectar. Sin centrar, la estructura buscada sería un subespacio afín, y la
proyección no sería lineal.

In [ ]:
# Datos concentrados alrededor de una direccion: un subespacio de dimension 1 los describe casi por completo.
rng = np.random.default_rng(4)
direccion = np.array([2.0, 1.0]) / np.sqrt(5)
t = rng.normal(scale=2.0, size=200)
datos = np.outer(t, direccion) + 0.25 * rng.normal(size=(200, 2))

centrados = datos - datos.mean(axis=0)
U, s, Vt = np.linalg.svd(centrados, full_matrices=False)
u1 = Vt[0]                                    # primera componente principal

# Proyeccion sobre el subespacio generado por u1: una matriz, porque es lineal.
P = np.outer(u1, u1)
proyectados = centrados @ P

print("primera componente principal:", np.round(u1, 4))
print("valores singulares:", np.round(s, 3))
print(f"varianza capturada por la primera direccion: {s[0] ** 2 / (s ** 2).sum():.1%}")

print("\nla proyeccion es lineal, luego P(x + y) == P(x) + P(y):",
      np.allclose((centrados[0] + centrados[1]) @ P, centrados[0] @ P + centrados[1] @ P))
print("y la imagen esta en el subespacio (idempotencia P @ P == P):", np.allclose(P @ P, P))

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(centrados[:, 0], centrados[:, 1], s=12, alpha=0.4, label="datos centrados")
ax.scatter(proyectados[:, 0], proyectados[:, 1], s=12, alpha=0.6, color="tab:red",
           label="proyectados sobre el subespacio")
lim = 6
ax.plot([-lim * u1[0], lim * u1[0]], [-lim * u1[1], lim * u1[1]], "--",
        color="black", lw=1, label="subespacio (dim 1)")
ax.set_xlim(-6, 6); ax.set_ylim(-6, 6); ax.set_aspect("equal")
ax.axhline(0, color="gray", lw=0.8); ax.axvline(0, color="gray", lw=0.8)
ax.grid(alpha=0.3); ax.legend(fontsize=8)
ax.set_title("Proyeccion sobre un subespacio de dimension 1")
plt.show()

---

## Tu turno

Los ejercicios son los propuestos en el documento `Matematicas_para_IA_01.pdf`. Sus respuestas están
en el PDF; el objetivo aquí es **decidir primero y verificar después**, apoyándose en las funciones
`verificar_grupo` y `criterio_subespacio` definidas arriba.

**Grupos**

**1.** ¿Es $(2\mathbb{Z}, +)$ —los enteros pares con la suma— un grupo? Verificar los cuatro
axiomas; para la cerradura y los inversos, apoyarse en la representación $2k$ con $k \in \mathbb{Z}$.

**2.** ¿Es $(\mathbb{R}^{2\times2}, \cdot)$ un grupo? Exhibir explícitamente una matriz que impida
que lo sea y comprobar con `np.linalg.det` y `np.linalg.inv` cuál axioma falla.

**3.** Verificar con `verificar_grupo` si $(\mathbb{Z}_6, \cdot)$ y $(\mathbb{Z}_7 \setminus \{0\}, \cdot)$
son grupos. Identificar qué distingue a $6$ de $7$ y enunciar la condición general.

**Espacios vectoriales**

**4.** Verificar la distributividad $\lambda(\mathbf{x} + \mathbf{y}) = \lambda\mathbf{x} +
\lambda\mathbf{y}$ en $\mathbb{R}^3$ con $\mathbf{x} = (1,0,2)$, $\mathbf{y} = (-1,3,1)$ y
$\lambda = 2$.

**5.** ¿Es el conjunto de polinomios de grado **exactamente** $2$, con la suma usual, un espacio
vectorial? Representar un polinomio por su vector de coeficientes, sumar $p(x) = x^2 + 1$ y
$q(x) = -x^2 + 3$, y examinar el grado del resultado. ¿Qué ocurre con los polinomios de grado
**menor o igual** que $2$?

**Subespacios**

**6.** ¿Es $U = \{(x,y,z) \in \mathbb{R}^3 : x + y + z = 0\}$ un subespacio de $\mathbb{R}^3$?
Verificar las tres condiciones y comprobar además que $U = N(A)$ con $A = [1, 1, 1]$.

**7.** ¿Es $U = \{(x,y) \in \mathbb{R}^2 : xy \geq 0\}$ un subespacio de $\mathbb{R}^2$? Buscar un
contraejemplo con `criterio_subespacio` partiendo de $(2,1)$ y $(-1,-3)$. Nótese que este conjunto
sí contiene al origen y sí es cerrado bajo el producto por escalar: el fallo está en otra condición.

**8.** Decidir si el conjunto de matrices $2 \times 2$ **simétricas** es un subespacio de
$\mathbb{R}^{2\times2}$, y hacer lo mismo con el conjunto de matrices **invertibles**. Uno de los
dos lo es y el otro no.

In [ ]:
# Tu codigo aqui.

# 1.

---

## Resumen

| Estructura | Qué exige | Ejemplo | Contraejemplo |
|---|---|---|---|
| Grupo | cerradura, asociatividad, neutro, inverso | $(\mathbb{Z}, +)$ | $(\mathbb{N}_0, +)$: faltan inversos |
| Grupo abeliano | lo anterior y conmutatividad | $(\mathbb{R}^n, +)$ | $GL(n,\mathbb{R})$: no conmuta |
| $GL(n, \mathbb{R})$ | matrices invertibles con el producto | $\det(A) \neq 0$ | las matrices singulares |
| Espacio vectorial | grupo abeliano y las reglas de los escalares | $\mathbb{R}^n$, $\mathbb{R}^{m\times n}$ | grado exactamente 2 |
| Subespacio | contener $\mathbf{0}$, cerradura bajo $+$ y escalar | $N(A)$, recta por el origen | solución de $A\mathbf{x}=\mathbf{b}$, $\mathbf{b} \neq \mathbf{0}$ |

Cuatro ideas para retener:

1. **La estructura es lo que autoriza las operaciones.** El descenso de gradiente y el promediado de
   modelos no son trucos de implementación: son operaciones de espacio vectorial, y existen porque
   el conjunto de parámetros es uno.
2. **Para refutar basta un contraejemplo; para demostrar no bastan mil ejemplos.** El código es
   concluyente en un sentido y sólo indicativo en el otro.
3. **El criterio de subespacio son tres comprobaciones.** Contener el cero, cerradura bajo escalar,
   cerradura bajo suma. Casi siempre falla la primera o la segunda.
4. **Pasar por el origen no es un detalle.** Es lo que separa un subespacio de su traslación, y la
   razón por la que PCA exige centrar los datos.

## Qué sigue

- La versión en script, más corta y sin explicaciones: [`01_vectores_y_espacios_vectoriales.py`](01_vectores_y_espacios_vectoriales.py)
- El documento de la sesión: `Matematicas_para_IA_01.pdf`
- El notebook anterior: [`00 · Propiedades de matrices`](00_propiedades_matrices.ipynb)
- El siguiente notebook: [`02 · Sistemas lineales, espacio nulo y rango`](02_sistemas_lineales_y_espacio_nulo.ipynb)
- Los demás módulos, en el [README del repositorio](../../README.md)

**Referencia.** Deisenroth, M. P., Faisal, A. A. y Ong, C. S. (2020). *Mathematics for Machine
Learning*. Cambridge University Press, sección 2.4.

---

*Material abierto bajo licencia MIT. ¿Encontraste un error o quieres aportar un ejercicio?
Lee [CONTRIBUTING.md](../../CONTRIBUTING.md).*